In [1]:
# CELDA 1 — Setup y carga de modelos
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os, joblib, warnings, unicodedata
import matplotlib.pyplot as plt
from datetime import datetime
warnings.filterwarnings('ignore')

PROJECT_PATH   = '/content/drive/MyDrive/HortifrutCostosImport'
PROCESSED_PATH = f'{PROJECT_PATH}/data/processed'
MODELS_PATH    = f'{PROJECT_PATH}/models'

print("⏳ Cargando modelos serializados...")

# === Regresión ===
modelo_regresion = joblib.load(f'{MODELS_PATH}/lightgbm_model.joblib')
modelo_p10 = joblib.load(f'{MODELS_PATH}/lgb_quantile_p10_cqr.joblib')
modelo_p50 = joblib.load(f'{MODELS_PATH}/lgb_quantile_p50_cqr.joblib')
modelo_p90 = joblib.load(f'{MODELS_PATH}/lgb_quantile_p90_cqr.joblib')
cqr_cal = joblib.load(f'{MODELS_PATH}/cqr_calibration.joblib')
Q_ALPHA = cqr_cal['Q_alpha']

# === Clasificación ===
modelo_clasif = joblib.load(f'{MODELS_PATH}/xgboost_classifier.joblib')
label_encoder  = joblib.load(f'{MODELS_PATH}/xgboost_label_encoder.joblib')

# === Clustering ===
modelo_cluster = joblib.load(f'{MODELS_PATH}/kmeans_k6.joblib')
cluster_prep   = joblib.load(f'{MODELS_PATH}/clustering_preprocessor.joblib')

# === Dataset histórico (necesario para calcular tarifa_historica de despachos nuevos) ===
hist = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_train.parquet')
hist = hist[hist['Importe_Total_PEN'] > 0].copy()

print(f"✓ Modelos cargados (regresión, clasificación, clustering)")
print(f"✓ Q_alpha de calibración CQR: S/ {Q_ALPHA:,.0f}")
print(f"✓ Histórico de referencia: {hist.shape[0]:,} facturas × {hist['Nro. Ope.'].nunique():,} operaciones")

Mounted at /content/drive
⏳ Cargando modelos serializados...
✓ Modelos cargados (regresión, clasificación, clustering)
✓ Q_alpha de calibración CQR: S/ 43
✓ Histórico de referencia: 11,181 facturas × 1,045 operaciones


In [2]:
# CELDA 2 — Reconstruir encoders y definir listas de features
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# Listas de features (mismo orden que en regresión/clasificación)
NUMERICAS = ['Cantidad de Bultos (BULKS)', 'Cantidad de Contenedores',
             'Peso Bruto (kg)', 'bultos_por_contenedor', 'peso_por_contenedor',
             'dias_desde_inicio', 'tarifa_historica', 'proveedor_frecuencia']

CATEGORICAS = ['Concepto Canónico', 'Proveedor_norm', 'ACREEDOR_norm',
               'AGENCIA DE ADUANA_norm', 'Proveedor Principal_norm',
               'incoterm_familia', 'Tipo de Contenedor', 'Modalidad (MODE Y TYPE)',
               'POL', 'POD (Puerto Destino)', 'País de Origen (POL)',
               'Delivery type', 'Final delivery', 'Moneda']

BINARIAS  = ['peso_disponible', 'tiene_proyecto', 'fecha_original', 'es_temporada_alta']
TEMPORALES = ['año', 'mes', 'trimestre', 'semana_año', 'dia_semana', 'Campaña']

# Filtrar a las que existen en el histórico
def filtrar(lista, cols):
    return [c for c in lista if c in cols]
NUMERICAS = filtrar(NUMERICAS, hist.columns)
CATEGORICAS = filtrar(CATEGORICAS, hist.columns)
BINARIAS = filtrar(BINARIAS, hist.columns)
TEMPORALES = filtrar(TEMPORALES, hist.columns)
FEATURES = NUMERICAS + CATEGORICAS + BINARIAS + TEMPORALES

# Pre-fill columnas 100% nulas (mismo tratamiento que en entrenamiento)
hist_features = hist[FEATURES].copy()
for col in NUMERICAS:
    if hist_features[col].notna().sum() == 0:
        hist_features[col] = 0

# Entrenar imputer + encoder
imputer_num = SimpleImputer(strategy='median')
imputer_num.fit(hist_features[NUMERICAS])

ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
ord_enc.fit(hist_features[CATEGORICAS].fillna('DESCONOCIDO').astype(str))

# Lista de conceptos canónicos para predicción
CONCEPTOS_CANONICOS = sorted(hist['Concepto Canónico'].dropna().unique().tolist())

# Lista de features para clustering (a nivel operación — distinto a regresión)
CLUSTER_NUM = ['peso_total', 'contenedores', 'bultos', 'costo_total_pen',
               'n_facturas', 'n_conceptos', 'n_proveedores']
CLUSTER_CAT = ['modalidad', 'incoterm', 'pol', 'pod', 'categoria', 'proveedor_principal']

print(f"✓ Features de regresión/clasificación: {len(FEATURES)}")
print(f"✓ Conceptos canónicos disponibles: {len(CONCEPTOS_CANONICOS)}")
for c in CONCEPTOS_CANONICOS:
    print(f"    • {c}")

✓ Features de regresión/clasificación: 32
✓ Conceptos canónicos disponibles: 13
    • AGENCIAMIENTO_ADUANA
    • DERECHOS_IMPUESTOS
    • DESCARGA
    • FITOSANITARIOS_SENASA
    • FLETE_INTERNACIONAL
    • HANDLING_PUERTO
    • INSPECCION_VERIFICACION
    • OTROS
    • SEGUROS
    • SERVICIOS_LOGISTICOS
    • SOBRESTADIA_ALMACENAJE
    • TRANSPORTE_T1_CALLAO_LIMA
    • TRANSPORTE_T2_LIMA_FUNDO


In [3]:
# CELDA 3 — Función auxiliar para tarifa histórica + normalización de texto
def normalizar_categorica(s):
    """Mayúsculas, sin tildes, sin puntos, espacios colapsados — igual que en entrenamiento."""
    if pd.isna(s) or s is None:
        return 'DESCONOCIDO'
    s = str(s).upper().strip()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    s = s.replace('.', '').replace(',', '')
    s = ' '.join(s.split())
    return s

def calcular_tarifa_historica(proveedor_norm, concepto_canonico):
    """
    Calcula tarifa esperada con cascada de fallbacks:
    1. Mediana del Proveedor × Concepto si hay ≥3 facturas históricas
    2. Mediana del Concepto si hay ≥3 facturas
    3. Mediana global
    """
    mask = (hist['Proveedor_norm'] == proveedor_norm) & (hist['Concepto Canónico'] == concepto_canonico)
    if mask.sum() >= 3:
        return hist.loc[mask, 'Importe_Total_PEN'].median()
    mask2 = hist['Concepto Canónico'] == concepto_canonico
    if mask2.sum() >= 3:
        return hist.loc[mask2, 'Importe_Total_PEN'].median()
    return hist['Importe_Total_PEN'].median()

def contar_proveedor_frecuencia(proveedor_norm):
    """Cuántas veces apareció el proveedor en el histórico."""
    return int((hist['Proveedor_norm'] == proveedor_norm).sum())

# Test rápido — cuánto se le ha cobrado históricamente a Hortifrut por agenciamiento de aduana
print("🧪 Tests de tarifa histórica:")
ejemplos = [
    ('AVM ADUANERA SAC', 'AGENCIAMIENTO_ADUANA'),
    ('IMUPESA', 'DESCARGA'),
    ('AIRSEALOG', 'FLETE_INTERNACIONAL'),
    ('PROVEEDOR_NUEVO_INVENTADO', 'HANDLING_PUERTO'),  # debería usar fallback
]
for prov, concepto in ejemplos:
    tarifa = calcular_tarifa_historica(prov, concepto)
    freq   = contar_proveedor_frecuencia(prov)
    print(f"  {prov:30} × {concepto:25} → S/ {tarifa:>8,.0f}  (proveedor visto {freq} veces)")

🧪 Tests de tarifa histórica:
  AVM ADUANERA SAC               × AGENCIAMIENTO_ADUANA      → S/      709  (proveedor visto 1783 veces)
  IMUPESA                        × DESCARGA                  → S/    1,677  (proveedor visto 0 veces)
  AIRSEALOG                      × FLETE_INTERNACIONAL       → S/    1,777  (proveedor visto 0 veces)
  PROVEEDOR_NUEVO_INVENTADO      × HANDLING_PUERTO           → S/      383  (proveedor visto 0 veces)


In [4]:
# CELDA 4 — Construir features para predicción
def construir_features(despacho):
    """
    Recibe un dict con los datos del despacho nuevo y devuelve un DataFrame
    listo para alimentar los modelos de regresión y clasificación.
    Una fila por concepto canónico (13 filas en total).
    """
    fecha_eta = pd.to_datetime(despacho.get('fecha_eta', datetime.now()))

    # Normalizar categóricas del input
    proveedor_norm           = normalizar_categorica(despacho.get('proveedor_servicio'))
    proveedor_principal_norm = normalizar_categorica(despacho.get('proveedor_principal'))
    acreedor_norm            = normalizar_categorica(despacho.get('acreedor') or despacho.get('proveedor_servicio'))
    agencia_norm             = normalizar_categorica(despacho.get('agencia_aduana'))

    contenedores = max(despacho.get('contenedores', 0), 0)
    bultos       = despacho.get('bultos', 0)
    peso_kg      = despacho.get('peso_kg', 0)

    filas = []
    for concepto in CONCEPTOS_CANONICOS:
        fila = {
            'Concepto Canónico':         concepto,
            'Proveedor_norm':            proveedor_norm,
            'ACREEDOR_norm':             acreedor_norm,
            'AGENCIA DE ADUANA_norm':    agencia_norm,
            'Proveedor Principal_norm':  proveedor_principal_norm,
            'Cantidad de Bultos (BULKS)': bultos,
            'Cantidad de Contenedores':   contenedores,
            'Peso Bruto (kg)':           peso_kg,
            'bultos_por_contenedor':     bultos / max(contenedores, 1),
            'peso_por_contenedor':       peso_kg / max(contenedores, 1),
            'dias_desde_inicio':         (fecha_eta - pd.Timestamp('2020-07-01')).days,
            'tarifa_historica':          calcular_tarifa_historica(proveedor_norm, concepto),
            'proveedor_frecuencia':      contar_proveedor_frecuencia(proveedor_norm),
            'incoterm_familia':          despacho.get('incoterm_familia', 'GRUPO_C'),
            'Tipo de Contenedor':        despacho.get('tipo_contenedor', '40 HC'),
            'Modalidad (MODE Y TYPE)':   despacho.get('modalidad', 'SEA / FCL'),
            'POL':                       despacho.get('pol', 'DESCONOCIDO'),
            'POD (Puerto Destino)':      despacho.get('pod', 'CALLAO'),
            'País de Origen (POL)':      despacho.get('pais_origen', 'DESCONOCIDO'),
            'Delivery type':             despacho.get('delivery_type', 'DIRECTO'),
            'Final delivery':            despacho.get('final_delivery', 'CHAO'),
            'Moneda':                    'USD',
            'peso_disponible':           1 if peso_kg > 0 else 0,
            'tiene_proyecto':            1 if despacho.get('proyecto') else 0,
            'fecha_original':            1,
            'es_temporada_alta':         1 if fecha_eta.month in [8,9,10,11,12] else 0,
            'año':                       fecha_eta.year,
            'mes':                       fecha_eta.month,
            'trimestre':                 fecha_eta.quarter,
            'semana_año':                int(fecha_eta.isocalendar().week),
            'dia_semana':                fecha_eta.dayofweek,
            'Campaña':                   fecha_eta.year,
        }
        filas.append(fila)

    df = pd.DataFrame(filas)[FEATURES]
    return df

# Test rápido — construir features para un despacho de prueba
despacho_test = {
    'proveedor_servicio':  'AVM ADUANERA',
    'proveedor_principal': 'PROJAR',
    'agencia_aduana':      'AVM ADUANERA',
    'acreedor':            'AVM ADUANERA',
    'pol':                 'VALENCIA',
    'pod':                 'CALLAO',
    'pais_origen':         'ESPAÑA',
    'modalidad':           'SEA / FCL',
    'incoterm_familia':    'GRUPO_C',
    'tipo_contenedor':     '40 HC',
    'contenedores':        2,
    'bultos':              48,
    'peso_kg':             18500,
    'fecha_eta':           '2026-08-15',
    'delivery_type':       'DIRECTO',
    'final_delivery':      'CHAO',
    'proyecto':            'M&S',
}
X_demo = construir_features(despacho_test)
print(f"✓ Features construidas: {X_demo.shape}")
print(f"\n👀 Vista de las primeras 3 filas (3 conceptos canónicos):")
X_demo.head(3)

✓ Features construidas: (13, 32)

👀 Vista de las primeras 3 filas (3 conceptos canónicos):


,Cantidad de Bultos (BULKS),Cantidad de Contenedores,Peso Bruto (kg),bultos_por_contenedor,peso_por_contenedor,dias_desde_inicio,tarifa_historica,proveedor_frecuencia,Concepto Canónico,Proveedor_norm,...,peso_disponible,tiene_proyecto,fecha_original,es_temporada_alta,año,mes,trimestre,semana_año,dia_semana,Campaña
0,48,2,18500,24.0,9250.0,2236,531.42480,0,AGENCIAMIENTO_ADUANA,AVM ADUANERA,...,1,1,1,1,2026,8,3,33,5,2026
1,48,2,18500,24.0,9250.0,2236,14287.00000,0,DERECHOS_IMPUESTOS,AVM ADUANERA,...,1,1,1,1,2026,8,3,33,5,2026
2,48,2,18500,24.0,9250.0,2236,1676.87839,0,DESCARGA,AVM ADUANERA,...,1,1,1,1,2026,8,3,33,5,2026


In [5]:
# CELDA 5 — Función principal predecir_despacho() — VERSIÓN ROBUSTA
MEDIANAS_NUMERICAS = {}
for col in NUMERICAS:
    if col in hist.columns and hist[col].notna().any():
        MEDIANAS_NUMERICAS[col] = hist[col].median()
    else:
        MEDIANAS_NUMERICAS[col] = 0

def preprocesar_para_modelos(X, features_a_usar):
    """Aplica imputación manual + encoding (independiente de los nombres usados en fit)."""
    X_proc = X[features_a_usar].copy()

    num_in_features = [c for c in NUMERICAS if c in features_a_usar]
    for col in num_in_features:
        if X_proc[col].isna().any() or X_proc[col].notna().sum() == 0:
            X_proc[col] = X_proc[col].fillna(MEDIANAS_NUMERICAS.get(col, 0))

    cat_in_features = [c for c in CATEGORICAS if c in features_a_usar]
    if cat_in_features:
        cat_data = X_proc[cat_in_features].fillna('DESCONOCIDO').astype(str)
        X_proc[cat_in_features] = ord_enc.transform(cat_data)

    for col in BINARIAS + TEMPORALES:
        if col in X_proc.columns:
            X_proc[col] = X_proc[col].fillna(0).astype(int)
    return X_proc


def _limpiar_valor(v, default):
    """Convierte NaN/None/strings vacíos a un valor por defecto."""
    if v is None: return default
    if isinstance(v, float) and pd.isna(v): return default
    if isinstance(v, str) and v.strip() == '': return default
    return v


def predecir_despacho(despacho):
    """
    Recibe los datos de un despacho nuevo y devuelve un diccionario con las
    predicciones de los tres tracks de ML (regresión, clasificación, clustering).
    """
    # 1) Features para regresión / clasificación
    X = construir_features(despacho)

    # --- 2) Regresión ---
    X_reg = preprocesar_para_modelos(X, FEATURES)
    pred_log     = modelo_regresion.predict(X_reg)
    pred_p10_log = modelo_p10.predict(X_reg)
    pred_p50_log = modelo_p50.predict(X_reg)
    pred_p90_log = modelo_p90.predict(X_reg)

    pred_pen     = np.expm1(pred_log)
    pred_p10_pen = np.maximum(np.expm1(pred_p10_log) - Q_ALPHA, 0)
    pred_p90_pen = np.expm1(pred_p90_log) + Q_ALPHA

    # --- 3) Clasificación (sin tarifa_historica) ---
    FEATURES_CLASIF = [c for c in FEATURES if c != 'tarifa_historica']
    X_clf = preprocesar_para_modelos(X, FEATURES_CLASIF)
    riesgo_enc = modelo_clasif.predict(X_clf)
    riesgo = label_encoder.inverse_transform(riesgo_enc)

    # --- 4) Clustering: limpiar TODOS los NaN antes de pasarlos a KMeans ---
    peso_kg_clean      = float(_limpiar_valor(despacho.get('peso_kg'), 0))
    contenedores_clean = float(_limpiar_valor(despacho.get('contenedores'), 0))
    bultos_clean       = float(_limpiar_valor(despacho.get('bultos'), 0))

    op_features = pd.DataFrame([{
        'peso_total':           peso_kg_clean,
        'contenedores':         contenedores_clean,
        'bultos':               bultos_clean,
        'costo_total_pen':      float(pred_pen.sum()),
        'n_facturas':           13.0,
        'n_conceptos':          13.0,
        'n_proveedores':        1.0,
        'modalidad':            str(_limpiar_valor(despacho.get('modalidad'), 'SEA / FCL')),
        'incoterm':             str(_limpiar_valor(despacho.get('incoterm_familia'), 'GRUPO_C')),
        'pol':                  str(_limpiar_valor(despacho.get('pol'), 'DESCONOCIDO')),
        'pod':                  str(_limpiar_valor(despacho.get('pod'), 'CALLAO')),
        'categoria':            str(_limpiar_valor(despacho.get('categoria'), '#N/D')),
        'proveedor_principal':  normalizar_categorica(despacho.get('proveedor_principal')),
    }])
    # Doble red de seguridad: fillna por si quedó algún NaN escondido
    op_num_cols = ['peso_total','contenedores','bultos','costo_total_pen','n_facturas','n_conceptos','n_proveedores']
    op_cat_cols = ['modalidad','incoterm','pol','pod','categoria','proveedor_principal']
    for c in op_num_cols:
        op_features[c] = pd.to_numeric(op_features[c], errors='coerce').fillna(0)
    for c in op_cat_cols:
        op_features[c] = op_features[c].fillna('DESCONOCIDO').astype(str)

    op_X = cluster_prep.transform(op_features)
    cluster_id = int(modelo_cluster.predict(op_X)[0])

    nombres_cluster = {
        0: "Marítimos europeos pequeños con alta cantidad de bultos",
        1: "Aéreos express bajo costo (courier Santiago-Lima)",
        2: "Aéreos de alto valor declarado",
        3: "Marítimos europeos de gran volumen (sustratos)",
        4: "Marítimos asiáticos multi-contenedor (Colombo-Callao)",
        5: "Aéreos rutinarios Santiago-Callao (perfil mayoritario)",
    }

    return {
        'operacion': despacho.get('id_despacho', 'NUEVO_DESPACHO'),
        'fecha_eta': despacho.get('fecha_eta'),
        'cluster_id': cluster_id,
        'cluster_nombre': nombres_cluster.get(cluster_id, 'Sin nombre'),
        'predicciones_por_concepto': pd.DataFrame({
            'concepto':           CONCEPTOS_CANONICOS,
            'costo_predicho_pen': pred_pen.round(0).astype(int),
            'intervalo_inferior': pred_p10_pen.round(0).astype(int),
            'intervalo_superior': pred_p90_pen.round(0).astype(int),
            'riesgo_desvio':      riesgo,
        }),
        'costo_total_estimado':   int(pred_pen.sum()),
        'costo_minimo_estimado':  int(pred_p10_pen.sum()),
        'costo_maximo_estimado':  int(pred_p90_pen.sum()),
    }

print("✓ Función predecir_despacho() definida (versión robusta contra NaN)")

✓ Función predecir_despacho() definida (versión robusta contra NaN)


In [6]:
# CELDA 6 — Ejemplo: predicción para un despacho realista
despacho_ejemplo = {
    'id_despacho':         '26-099-HPER',
    'proveedor_servicio':  'AIRSEALOG',
    'proveedor_principal': 'PROJAR',          # importador de sustratos español
    'agencia_aduana':      'AVM ADUANERA',
    'acreedor':            'AVM ADUANERA',
    'pol':                 'VALENCIA',
    'pod':                 'CALLAO',
    'pais_origen':         'ESPAÑA',
    'modalidad':           'SEA / FCL',
    'incoterm_familia':    'GRUPO_C',
    'tipo_contenedor':     '40 HC',
    'contenedores':        2,
    'bultos':              48,
    'peso_kg':             18500,
    'fecha_eta':           '2026-08-15',
    'delivery_type':       'DIRECTO',
    'final_delivery':      'CHAO',
    'proyecto':            'M&S',
    'categoria':           'SUSTRATOS',
}

resultado = predecir_despacho(despacho_ejemplo)

# === Imprimir reporte tipo "informe contable" ===
print("="*75)
print(f"📋 INFORME DE PROVISIÓN PREDICTIVA — DESPACHO {resultado['operacion']}")
print("="*75)
print(f"ETA del despacho: {resultado['fecha_eta']}")
print(f"Perfil operativo: Cluster {resultado['cluster_id']} — {resultado['cluster_nombre']}")
print()
print(f"💰 COSTO TOTAL ESTIMADO: S/ {resultado['costo_total_estimado']:,}")
print(f"   Rango con 80% de confianza: S/ {resultado['costo_minimo_estimado']:,}  —  S/ {resultado['costo_maximo_estimado']:,}")
print()
print("📊 DESGLOSE POR CONCEPTO CANÓNICO")
print("-"*75)
df_pred = resultado['predicciones_por_concepto'].copy()
df_pred = df_pred.sort_values('costo_predicho_pen', ascending=False)
# Formatear como tabla
print(f"{'Concepto Canónico':<28} {'Predicho':>10} {'Mín P10':>10} {'Máx P90':>10}  {'Riesgo':<8}")
print("-"*75)
for _, row in df_pred.iterrows():
    riesgo_icon = {'BAJO':'✓','MEDIO':'⚠','ALTO':'⚠⚠'}.get(row['riesgo_desvio'], '?')
    print(f"{row['concepto']:<28} S/{row['costo_predicho_pen']:>8,}  S/{row['intervalo_inferior']:>8,}  S/{row['intervalo_superior']:>8,}  {riesgo_icon} {row['riesgo_desvio']}")
print("-"*75)
print(f"{'TOTAL':<28} S/{resultado['costo_total_estimado']:>8,}  S/{resultado['costo_minimo_estimado']:>8,}  S/{resultado['costo_maximo_estimado']:>8,}")
print("="*75)

📋 INFORME DE PROVISIÓN PREDICTIVA — DESPACHO 26-099-HPER
ETA del despacho: 2026-08-15
Perfil operativo: Cluster 5 — Aéreos rutinarios Santiago-Callao (perfil mayoritario)

💰 COSTO TOTAL ESTIMADO: S/ 38,006
   Rango con 80% de confianza: S/ 8,883  —  S/ 115,330

📊 DESGLOSE POR CONCEPTO CANÓNICO
---------------------------------------------------------------------------
Concepto Canónico              Predicho    Mín P10    Máx P90  Riesgo  
---------------------------------------------------------------------------
TRANSPORTE_T2_LIMA_FUNDO     S/   7,692  S/   1,864  S/  10,825  ⚠⚠ ALTO
SOBRESTADIA_ALMACENAJE       S/   6,823  S/   1,051  S/  15,320  ⚠⚠ ALTO
SERVICIOS_LOGISTICOS         S/   6,302  S/   1,354  S/  13,201  ⚠⚠ ALTO
TRANSPORTE_T1_CALLAO_LIMA    S/   4,175  S/     761  S/   7,832  ⚠⚠ ALTO
DERECHOS_IMPUESTOS           S/   4,012  S/   1,308  S/  28,255  ⚠⚠ ALTO
DESCARGA                     S/   3,007  S/     635  S/  11,307  ⚠⚠ ALTO
FLETE_INTERNACIONAL          S/   2,319  S/

In [7]:
# CELDA 7 — Validar la función con 3 operaciones reales del test set
test = pd.read_parquet(f'{PROCESSED_PATH}/dataset_modelable_test.parquet')
test = test[test['Importe_Total_PEN'] > 0].copy()

# Tomar 3 operaciones distintas al azar (semilla fija)
np.random.seed(42)
ops_muestra = np.random.choice(test['Nro. Ope.'].unique(), 3, replace=False)

print("🔍 VALIDACIÓN: predicción vs valor real para 3 operaciones del test set\n")

for op_id in ops_muestra:
    sub = test[test['Nro. Ope.'] == op_id]
    primera = sub.iloc[0]

    despacho_real = {
        'id_despacho':         op_id,
        'proveedor_servicio':  primera.get('Proveedor_norm'),
        'proveedor_principal': primera.get('Proveedor Principal_norm'),
        'agencia_aduana':      primera.get('AGENCIA DE ADUANA_norm'),
        'pol':                 primera.get('POL'),
        'pod':                 primera.get('POD (Puerto Destino)'),
        'modalidad':           primera.get('Modalidad (MODE Y TYPE)'),
        'incoterm_familia':    primera.get('incoterm_familia'),
        'contenedores':        primera.get('Cantidad de Contenedores', 0),
        'bultos':              primera.get('Cantidad de Bultos (BULKS)', 0),
        'peso_kg':             primera.get('Peso Bruto (kg)', 0),
        'fecha_eta':           primera.get('Fecha_Imputada'),
    }

    res = predecir_despacho(despacho_real)
    costo_real_operacion = sub['Importe_Total_PEN'].sum()
    error_abs = abs(res['costo_total_estimado'] - costo_real_operacion)
    error_pct = error_abs / costo_real_operacion * 100 if costo_real_operacion else 0

    print(f"OPERACIÓN {op_id}")
    print(f"  Costo REAL del despacho:      S/ {costo_real_operacion:>10,.0f}")
    print(f"  Costo PREDICHO:               S/ {res['costo_total_estimado']:>10,}")
    print(f"  Intervalo del 80%:            S/ {res['costo_minimo_estimado']:>10,} — S/ {res['costo_maximo_estimado']:,}")
    print(f"  Cluster asignado:             {res['cluster_id']} ({res['cluster_nombre']})")
    print(f"  Error absoluto: S/ {error_abs:,.0f}  ({error_pct:.1f}% del real)")
    dentro = res['costo_minimo_estimado'] <= costo_real_operacion <= res['costo_maximo_estimado']
    print(f"  ¿Valor real dentro del intervalo del 80%? {'✓ SÍ' if dentro else '✗ NO'}")
    print()

🔍 VALIDACIÓN: predicción vs valor real para 3 operaciones del test set

OPERACIÓN 24-141-HPER
  Costo REAL del despacho:      S/     41,425
  Costo PREDICHO:               S/     39,759
  Intervalo del 80%:            S/     20,815 — S/ 78,277
  Cluster asignado:             5 (Aéreos rutinarios Santiago-Callao (perfil mayoritario))
  Error absoluto: S/ 1,666  (4.0% del real)
  ¿Valor real dentro del intervalo del 80%? ✓ SÍ

OPERACIÓN 25-008-HPER
  Costo REAL del despacho:      S/      6,956
  Costo PREDICHO:               S/     12,697
  Intervalo del 80%:            S/      4,532 — S/ 36,644
  Cluster asignado:             5 (Aéreos rutinarios Santiago-Callao (perfil mayoritario))
  Error absoluto: S/ 5,741  (82.5% del real)
  ¿Valor real dentro del intervalo del 80%? ✓ SÍ

OPERACIÓN 25-097-HPER
  Costo REAL del despacho:      S/    169,649
  Costo PREDICHO:               S/    107,949
  Intervalo del 80%:            S/     29,312 — S/ 213,595
  Cluster asignado:             3 (Marít

In [8]:
# CELDA 8 — Fuzzy matching de proveedores
from difflib import get_close_matches

# Precomputar lista de proveedores únicos del histórico (para búsquedas rápidas)
PROVEEDORES_HIST = sorted(hist['Proveedor_norm'].dropna().unique().tolist())
print(f"Total proveedores únicos en histórico: {len(PROVEEDORES_HIST)}")

def fuzzy_match_proveedor(nombre_input, umbral=0.7):
    """
    Busca el proveedor del histórico más parecido al nombre dado.
    Si encuentra una coincidencia con similitud > umbral, devuelve el nombre histórico.
    Si no, devuelve el nombre original normalizado.
    """
    if not nombre_input or nombre_input == 'DESCONOCIDO':
        return 'DESCONOCIDO'

    # Primero intentar match exacto (más rápido)
    if nombre_input in PROVEEDORES_HIST:
        return nombre_input

    # Si no, buscar el más parecido
    candidatos = get_close_matches(nombre_input, PROVEEDORES_HIST, n=1, cutoff=umbral)
    if candidatos:
        return candidatos[0]
    return nombre_input

# Versión mejorada de calcular_tarifa_historica que usa fuzzy matching
def calcular_tarifa_historica_v2(proveedor_norm, concepto_canonico, verbose=False):
    """Versión con fuzzy matching: busca el proveedor más parecido si el exacto no existe."""
    # Intentar primero con el nombre exacto
    mask = (hist['Proveedor_norm'] == proveedor_norm) & (hist['Concepto Canónico'] == concepto_canonico)
    if mask.sum() >= 3:
        return hist.loc[mask, 'Importe_Total_PEN'].median(), proveedor_norm

    # Si no hay match, buscar fuzzy
    proveedor_fuzzy = fuzzy_match_proveedor(proveedor_norm)
    if proveedor_fuzzy != proveedor_norm:
        if verbose:
            print(f"   ℹ Fuzzy match: '{proveedor_norm}' → '{proveedor_fuzzy}'")
        mask = (hist['Proveedor_norm'] == proveedor_fuzzy) & (hist['Concepto Canónico'] == concepto_canonico)
        if mask.sum() >= 3:
            return hist.loc[mask, 'Importe_Total_PEN'].median(), proveedor_fuzzy

    # Fallback por concepto
    mask2 = hist['Concepto Canónico'] == concepto_canonico
    if mask2.sum() >= 3:
        return hist.loc[mask2, 'Importe_Total_PEN'].median(), 'FALLBACK_CONCEPTO'

    return hist['Importe_Total_PEN'].median(), 'FALLBACK_GLOBAL'

# Tests
print("\n🧪 Tests de fuzzy matching:")
for prov in ['AIRSEALOG', 'IMUPESA', 'AVM ADUANERA', 'PROVEEDOR INVENTADO']:
    match = fuzzy_match_proveedor(prov)
    tarifa, fuente = calcular_tarifa_historica_v2(prov, 'AGENCIAMIENTO_ADUANA', verbose=True)
    print(f"  '{prov}' → match: '{match}' | tarifa: S/ {tarifa:,.0f} | fuente: {fuente}")

Total proveedores únicos en histórico: 134

🧪 Tests de fuzzy matching:
  'AIRSEALOG' → match: 'AIRSEALOG' | tarifa: S/ 531 | fuente: FALLBACK_CONCEPTO
  'IMUPESA' → match: 'IMUPESA' | tarifa: S/ 531 | fuente: FALLBACK_CONCEPTO
   ℹ Fuzzy match: 'AVM ADUANERA' → 'AVM ADUANERA SAC'
  'AVM ADUANERA' → match: 'AVM ADUANERA SAC' | tarifa: S/ 709 | fuente: AVM ADUANERA SAC
  'PROVEEDOR INVENTADO' → match: 'PROVEEDOR INVENTADO' | tarifa: S/ 531 | fuente: FALLBACK_CONCEPTO


In [9]:
# CELDA 9 — Cluster ajustado (sin usar costo predicho)

# Cargar las medianas del costo por cluster del histórico (para inferir el cluster sin predicción)
df_ops_hist = pd.read_parquet(f'{PROCESSED_PATH}/operaciones_con_clusters.parquet')

# Calcular el "costo típico" de cada cluster (mediana histórica) — solo se usa una vez como referencia
COSTO_TIPICO_CLUSTER = df_ops_hist.groupby('cluster_kmeans')['costo_total_pen'].median().to_dict()
print(f"📊 Costo mediano por cluster (histórico):")
for c, v in sorted(COSTO_TIPICO_CLUSTER.items()):
    print(f"   Cluster {c}: S/ {v:>10,.0f}")

def predecir_cluster_v2(despacho):
    """
    Asigna el cluster basándose en features físicas y operativas del despacho.
    No usa el costo predicho (evita circularidad).
    """
    peso_kg      = float(_limpiar_valor(despacho.get('peso_kg'), 0))
    contenedores = float(_limpiar_valor(despacho.get('contenedores'), 0))
    bultos       = float(_limpiar_valor(despacho.get('bultos'), 0))

    # Para el costo_total, usamos un proxy basado en el HISTORICO de operaciones similares
    # (no en la predicción del modelo)
    # Buscamos operaciones con modalidad + POL + POD similares y tomamos su costo mediano
    modalidad = str(_limpiar_valor(despacho.get('modalidad'), 'SEA / FCL'))
    pol       = str(_limpiar_valor(despacho.get('pol'), 'DESCONOCIDO'))
    pod       = str(_limpiar_valor(despacho.get('pod'), 'CALLAO'))

    # Buscar operaciones similares en el histórico
    mask = (
        (df_ops_hist['modalidad'].astype(str) == modalidad) &
        (df_ops_hist['pol'].astype(str) == pol) &
        (df_ops_hist['pod'].astype(str) == pod)
    )
    if mask.sum() >= 3:
        costo_proxy = df_ops_hist.loc[mask, 'costo_total_pen'].median()
    else:
        # Fallback: por modalidad sola
        mask2 = df_ops_hist['modalidad'].astype(str) == modalidad
        costo_proxy = df_ops_hist.loc[mask2, 'costo_total_pen'].median() if mask2.sum() > 0 \
                     else df_ops_hist['costo_total_pen'].median()

    # Construir features para el cluster
    op_features = pd.DataFrame([{
        'peso_total':           peso_kg,
        'contenedores':         contenedores,
        'bultos':               bultos,
        'costo_total_pen':      float(costo_proxy),  # ← proxy histórico, no predicción
        'n_facturas':           13.0,
        'n_conceptos':          13.0,
        'n_proveedores':        1.0,
        'modalidad':            modalidad,
        'incoterm':             str(_limpiar_valor(despacho.get('incoterm_familia'), 'GRUPO_C')),
        'pol':                  pol,
        'pod':                  pod,
        'categoria':            str(_limpiar_valor(despacho.get('categoria'), '#N/D')),
        'proveedor_principal':  normalizar_categorica(despacho.get('proveedor_principal')),
    }])

    # Limpieza defensiva
    for c in ['peso_total','contenedores','bultos','costo_total_pen','n_facturas','n_conceptos','n_proveedores']:
        op_features[c] = pd.to_numeric(op_features[c], errors='coerce').fillna(0)
    for c in ['modalidad','incoterm','pol','pod','categoria','proveedor_principal']:
        op_features[c] = op_features[c].fillna('DESCONOCIDO').astype(str)

    op_X = cluster_prep.transform(op_features)
    return int(modelo_cluster.predict(op_X)[0])

# Test: el despacho de sustratos europeos ahora DEBERÍA caer en el cluster 3
despacho_test = {
    'peso_kg': 18500, 'contenedores': 2, 'bultos': 48,
    'modalidad': 'SEA / FCL', 'incoterm_familia': 'GRUPO_C',
    'pol': 'VALENCIA', 'pod': 'CALLAO',
    'categoria': 'SUSTRATOS', 'proveedor_principal': 'PROJAR'
}
cluster_id = predecir_cluster_v2(despacho_test)
nombres_cluster = {
    0: "Marítimos europeos pequeños con alta cantidad de bultos",
    1: "Aéreos express bajo costo (courier Santiago-Lima)",
    2: "Aéreos de alto valor declarado",
    3: "Marítimos europeos de gran volumen (sustratos)",
    4: "Marítimos asiáticos multi-contenedor (Colombo-Callao)",
    5: "Aéreos rutinarios Santiago-Callao (perfil mayoritario)",
}
print(f"\n🧪 Test del cluster v2 con despacho de sustratos europeos:")
print(f"   Cluster asignado: {cluster_id} → {nombres_cluster.get(cluster_id, 'Sin nombre')}")
print(f"   (Esperado: 0 o 3 — ambos son perfiles marítimos europeos)")

📊 Costo mediano por cluster (histórico):
   Cluster 0: S/    115,040
   Cluster 1: S/      1,170
   Cluster 2: S/    381,554
   Cluster 3: S/    110,556
   Cluster 4: S/    103,966
   Cluster 5: S/     28,333

🧪 Test del cluster v2 con despacho de sustratos europeos:
   Cluster asignado: 5 → Aéreos rutinarios Santiago-Callao (perfil mayoritario)
   (Esperado: 0 o 3 — ambos son perfiles marítimos europeos)


In [10]:
# CELDA 10 — Formulario interactivo para probar despachos
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widgets de entrada
w_id            = widgets.Text(value='DEMO-001', description='ID Despacho:')
w_proveedor     = widgets.Text(value='AIRSEALOG', description='Proveedor Servicio:')
w_principal     = widgets.Text(value='PROJAR', description='Proveedor Principal:')
w_agencia       = widgets.Text(value='AVM ADUANERA', description='Agencia Aduana:')
w_pol           = widgets.Dropdown(
    options=['VALENCIA','SAN ANTONIO','SANTIAGO','BARCELONA','COLOMBO','MIAMI','BRUSELAS','LEIXOES'],
    value='VALENCIA', description='POL (origen):')
w_pod           = widgets.Dropdown(options=['CALLAO','LIMA'], value='CALLAO', description='POD (destino):')
w_modalidad     = widgets.Dropdown(options=['SEA / FCL','SEA / LCL','AIR / AIR'], value='SEA / FCL', description='Modalidad:')
w_incoterm      = widgets.Dropdown(options=['GRUPO_C','GRUPO_E','GRUPO_F','GRUPO_D'], value='GRUPO_C', description='INCOTERM:')
w_contenedores  = widgets.IntSlider(value=2, min=0, max=10, description='Contenedores:')
w_bultos        = widgets.IntSlider(value=48, min=0, max=2000, step=10, description='Bultos:')
w_peso          = widgets.IntSlider(value=18500, min=0, max=100000, step=500, description='Peso (kg):')
w_categoria     = widgets.Dropdown(options=['SUSTRATOS','PLANTAS','AGROQUIMICOS','SPAREPARTS','MAQUINARIA','E&E','#N/D'],
                                    value='SUSTRATOS', description='Categoría:')
w_fecha         = widgets.Text(value='2026-08-15', description='Fecha ETA:')

# Botón y output
boton = widgets.Button(description='🔮 Predecir', button_style='primary', layout=widgets.Layout(width='200px'))
output = widgets.Output()

def on_predict(b):
    with output:
        clear_output()
        despacho = {
            'id_despacho':         w_id.value,
            'proveedor_servicio':  w_proveedor.value,
            'proveedor_principal': w_principal.value,
            'agencia_aduana':      w_agencia.value,
            'acreedor':            w_agencia.value,
            'pol':                 w_pol.value,
            'pod':                 w_pod.value,
            'modalidad':           w_modalidad.value,
            'incoterm_familia':    w_incoterm.value,
            'contenedores':        w_contenedores.value,
            'bultos':              w_bultos.value,
            'peso_kg':             w_peso.value,
            'categoria':           w_categoria.value,
            'fecha_eta':           w_fecha.value,
            'tipo_contenedor':     '40 HC',
            'pais_origen':         w_pol.value,
        }
        res = predecir_despacho(despacho)
        res['cluster_id'] = asignar_cluster_heuristico(despacho)
        res['cluster_nombre'] = nombres_cluster.get(res['cluster_id'], 'Sin nombre')

        print(f"💰 COSTO TOTAL ESTIMADO: S/ {res['costo_total_estimado']:,}")
        print(f"   Rango 80% confianza:    S/ {res['costo_minimo_estimado']:,} — S/ {res['costo_maximo_estimado']:,}")
        print(f"   Perfil operativo:       Cluster {res['cluster_id']} — {res['cluster_nombre']}")
        print(f"\n📊 Desglose por concepto:")
        print(res['predicciones_por_concepto'].sort_values('costo_predicho_pen', ascending=False).to_string(index=False))

boton.on_click(on_predict)

# Layout: dos columnas
col_izq = widgets.VBox([w_id, w_proveedor, w_principal, w_agencia, w_pol, w_pod])
col_der = widgets.VBox([w_modalidad, w_incoterm, w_contenedores, w_bultos, w_peso, w_categoria, w_fecha])
formulario = widgets.HBox([col_izq, col_der])

display(widgets.VBox([
    widgets.HTML("<h3>🛠️ Simulador de Predicción de Costos de Importación</h3>"),
    widgets.HTML("<p>Ajustá los valores del despacho y presioná <b>Predecir</b> para ver el resultado.</p>"),
    formulario,
    boton,
    output
]))

In [11]:
# CELDA 11 — Asignación heurística de cluster (reemplaza la asignación de K-Means en el demo)

def asignar_cluster_heuristico(despacho):
    """
    Asigna el cluster usando reglas simples basadas en los perfiles que el
    K-Means descubrió. Más interpretable y predecible que dejar al modelo
    decidir con poca data.
    """
    modalidad = str(despacho.get('modalidad', '')).upper()
    pol       = str(despacho.get('pol', '')).upper()
    pod       = str(despacho.get('pod', '')).upper()
    peso      = float(_limpiar_valor(despacho.get('peso_kg'), 0))
    contenedores = float(_limpiar_valor(despacho.get('contenedores'), 0))
    bultos    = float(_limpiar_valor(despacho.get('bultos'), 0))

    # === Aéreos ===
    if 'AIR' in modalidad:
        if 'SANTIAGO' in pol and peso < 3000:
            return 1   # Aéreos express courier Santiago-Lima
        if peso > 5000 or bultos > 100:
            return 2   # Aéreos de alto valor declarado
        return 5       # Aéreos rutinarios (perfil mayoritario)

    # === Marítimos europeos ===
    EUR_POLS = {'VALENCIA','BARCELONA','LEIXOES','HAMBURG','ROTTERDAM','GENOA'}
    if any(p in pol for p in EUR_POLS):
        if peso > 50000 or contenedores >= 3 or bultos > 100:
            return 3   # Marítimos europeos de gran volumen (sustratos)
        return 0       # Marítimos europeos pequeños alta cantidad bultos

    # === Marítimos asiáticos ===
    ASIA_POLS = {'COLOMBO','SHANGHAI','NINGBO','BUSAN','HONG KONG','TAIWAN'}
    if any(p in pol for p in ASIA_POLS):
        return 4       # Marítimos asiáticos multi-contenedor

    # === Default: aéreos rutinarios ===
    return 5

# Reemplazamos la lógica del cluster en el demo: actualizar la función de los widgets
nombres_cluster_dict = {
    0: "Marítimos europeos pequeños con alta cantidad de bultos",
    1: "Aéreos express bajo costo (courier Santiago-Lima)",
    2: "Aéreos de alto valor declarado",
    3: "Marítimos europeos de gran volumen (sustratos)",
    4: "Marítimos asiáticos multi-contenedor (Colombo-Callao)",
    5: "Aéreos rutinarios Santiago-Callao (perfil mayoritario)",
}

# Tests con casos representativos
casos_test = [
    {'nombre': 'Sustratos europeos grande', 'despacho': {
        'modalidad': 'SEA / FCL', 'pol': 'VALENCIA', 'peso_kg': 18500,
        'contenedores': 2, 'bultos': 48
    }},
    {'nombre': 'Repuestos courier Chile', 'despacho': {
        'modalidad': 'AIR / AIR', 'pol': 'SANTIAGO', 'peso_kg': 100,
        'contenedores': 0, 'bultos': 1
    }},
    {'nombre': 'Plantas aéreas alto valor', 'despacho': {
        'modalidad': 'AIR / AIR', 'pol': 'SANTIAGO', 'peso_kg': 8000,
        'contenedores': 0, 'bultos': 200
    }},
    {'nombre': 'Contenedores asiáticos', 'despacho': {
        'modalidad': 'SEA / FCL', 'pol': 'COLOMBO', 'peso_kg': 22000,
        'contenedores': 3, 'bultos': 500
    }},
    {'nombre': 'Sustratos europeos chico', 'despacho': {
        'modalidad': 'SEA / FCL', 'pol': 'VALENCIA', 'peso_kg': 10000,
        'contenedores': 1, 'bultos': 1500
    }},
]

print("🧪 Tests de la asignación heurística de cluster:\n")
for caso in casos_test:
    c_id = asignar_cluster_heuristico(caso['despacho'])
    print(f"  {caso['nombre']:30} → Cluster {c_id}: {nombres_cluster_dict[c_id]}")

print("\n✓ Ahora podés re-ejecutar la celda 10 (los widgets) y al presionar Predecir,")
print("  el cluster se asignará con la heurística — más predecible y correcto.")

🧪 Tests de la asignación heurística de cluster:

  Sustratos europeos grande      → Cluster 0: Marítimos europeos pequeños con alta cantidad de bultos
  Repuestos courier Chile        → Cluster 1: Aéreos express bajo costo (courier Santiago-Lima)
  Plantas aéreas alto valor      → Cluster 2: Aéreos de alto valor declarado
  Contenedores asiáticos         → Cluster 4: Marítimos asiáticos multi-contenedor (Colombo-Callao)
  Sustratos europeos chico       → Cluster 3: Marítimos europeos de gran volumen (sustratos)

✓ Ahora podés re-ejecutar la celda 10 (los widgets) y al presionar Predecir,
  el cluster se asignará con la heurística — más predecible y correcto.


In [12]:
# CELDA 12 — Detectar el proveedor típico de cada concepto canónico
# y usarlo en lugar del proveedor único ingresado en el formulario

# Para cada concepto canónico, encontrar el proveedor que más facturó históricamente
proveedor_top_por_concepto = (
    hist.groupby(['Concepto Canónico', 'Proveedor_norm'])
        .size()
        .reset_index(name='count')
        .sort_values(['Concepto Canónico', 'count'], ascending=[True, False])
        .groupby('Concepto Canónico')
        .first()
        .reset_index()
)
PROVEEDOR_TIPICO = dict(zip(
    proveedor_top_por_concepto['Concepto Canónico'],
    proveedor_top_por_concepto['Proveedor_norm']
))

print("📋 Proveedor más frecuente por concepto canónico:")
for concepto, prov in PROVEEDOR_TIPICO.items():
    n = ((hist['Concepto Canónico']==concepto) & (hist['Proveedor_norm']==prov)).sum()
    print(f"  {concepto:30} → {prov:30} ({n:>4} facturas históricas)")


# Versión mejorada de construir_features que usa el proveedor típico por concepto
def construir_features_v2(despacho):
    """Como construir_features pero usa el proveedor típico de cada concepto."""
    fecha_eta = pd.to_datetime(despacho.get('fecha_eta', datetime.now()))

    # Proveedor "general" que ingresa el usuario (lo usamos como fallback)
    proveedor_input           = normalizar_categorica(despacho.get('proveedor_servicio'))
    proveedor_principal_norm  = normalizar_categorica(despacho.get('proveedor_principal'))

    contenedores = max(despacho.get('contenedores', 0), 0)
    bultos       = despacho.get('bultos', 0)
    peso_kg      = despacho.get('peso_kg', 0)

    filas = []
    for concepto in CONCEPTOS_CANONICOS:
        # El proveedor para este concepto: el típico histórico si existe, sino el input
        proveedor_para_concepto = PROVEEDOR_TIPICO.get(concepto, proveedor_input)

        fila = {
            'Concepto Canónico':         concepto,
            'Proveedor_norm':            proveedor_para_concepto,
            'ACREEDOR_norm':             proveedor_para_concepto,
            'AGENCIA DE ADUANA_norm':    normalizar_categorica(despacho.get('agencia_aduana')),
            'Proveedor Principal_norm':  proveedor_principal_norm,
            'Cantidad de Bultos (BULKS)': bultos,
            'Cantidad de Contenedores':   contenedores,
            'Peso Bruto (kg)':           peso_kg,
            'bultos_por_contenedor':     bultos / max(contenedores, 1),
            'peso_por_contenedor':       peso_kg / max(contenedores, 1),
            'dias_desde_inicio':         (fecha_eta - pd.Timestamp('2020-07-01')).days,
            'tarifa_historica':          calcular_tarifa_historica(proveedor_para_concepto, concepto),
            'proveedor_frecuencia':      contar_proveedor_frecuencia(proveedor_para_concepto),
            'incoterm_familia':          despacho.get('incoterm_familia', 'GRUPO_C'),
            'Tipo de Contenedor':        despacho.get('tipo_contenedor', '40 HC'),
            'Modalidad (MODE Y TYPE)':   despacho.get('modalidad', 'SEA / FCL'),
            'POL':                       despacho.get('pol', 'DESCONOCIDO'),
            'POD (Puerto Destino)':      despacho.get('pod', 'CALLAO'),
            'País de Origen (POL)':      despacho.get('pais_origen', 'DESCONOCIDO'),
            'Delivery type':             despacho.get('delivery_type', 'DIRECTO'),
            'Final delivery':            despacho.get('final_delivery', 'CHAO'),
            'Moneda':                    'USD',
            'peso_disponible':           1 if peso_kg > 0 else 0,
            'tiene_proyecto':            1 if despacho.get('proyecto') else 0,
            'fecha_original':            1,
            'es_temporada_alta':         1 if fecha_eta.month in [8,9,10,11,12] else 0,
            'año':                       fecha_eta.year,
            'mes':                       fecha_eta.month,
            'trimestre':                 fecha_eta.quarter,
            'semana_año':                int(fecha_eta.isocalendar().week),
            'dia_semana':                fecha_eta.dayofweek,
            'Campaña':                   fecha_eta.year,
        }
        filas.append(fila)

    return pd.DataFrame(filas)[FEATURES]


# Reemplazamos la función original para que los widgets usen la nueva
construir_features_original = construir_features
construir_features = construir_features_v2

print("\n✓ Ahora cada concepto usa el proveedor típico histórico.")
print("✓ Re-ejecutá la celda 10 (widgets) y prob\u00e1 los 3 ejemplos otra vez.")
print("  Los costos predichos deberían acercarse mucho más a los valores reales.")

📋 Proveedor más frecuente por concepto canónico:
  AGENCIAMIENTO_ADUANA           → AVM ADUANERA SAC               ( 349 facturas históricas)
  DERECHOS_IMPUESTOS             → SUNAT                          (1871 facturas históricas)
  DESCARGA                       → INVERSIONES MARITIMAS UNIVERSALES PERU SA ( 459 facturas históricas)
  FITOSANITARIOS_SENASA          → SERVICIO NACIONAL DE SANIDAD AGRARIA ( 844 facturas históricas)
  FLETE_INTERNACIONAL            → AIRSEALOGISTICS SAC            ( 386 facturas históricas)
  HANDLING_PUERTO                → AIRSEALOGISTICS SAC            ( 148 facturas históricas)
  INSPECCION_VERIFICACION        → SERVICIO NACIONAL DE SANIDAD AGRARIA ( 405 facturas históricas)
  OTROS                          → SUNAT                          (  44 facturas históricas)
  SEGUROS                        → LA POSITIVA SEGUROS Y REASEGUROS SAA ( 393 facturas históricas)
  SERVICIOS_LOGISTICOS           → AVM ADUANERA SAC               ( 316 facturas hist

In [13]:
# DEBUG — probar la heurística directamente con los inputs del Ejemplo 1
test_despacho = {
    'modalidad': 'AIR / AIR',
    'pol': 'SANTIAGO',
    'pod': 'CALLAO',
    'peso_kg': 50,
    'contenedores': 0,
    'bultos': 1,
    'incoterm_familia': 'GRUPO_E',
}

resultado_directo = asignar_cluster_heuristico(test_despacho)
print(f"Resultado heurística directo: cluster {resultado_directo}")
print(f"Esperado: 1")
print()
print("Trace paso a paso:")
mod = str(test_despacho.get('modalidad', '')).upper()
pol = str(test_despacho.get('pol', '')).upper()
peso = float(test_despacho.get('peso_kg', 0))
bultos = float(test_despacho.get('bultos', 0))
print(f"  modalidad.upper() = '{mod}'   | 'AIR' in modalidad: {'AIR' in mod}")
print(f"  pol.upper() = '{pol}'         | 'SANTIAGO' in pol: {'SANTIAGO' in pol}")
print(f"  peso = {peso}                 | peso < 3000: {peso < 3000}")
print(f"  bultos = {bultos}             | bultos > 100: {bultos > 100}")

Resultado heurística directo: cluster 1
Esperado: 1

Trace paso a paso:
  modalidad.upper() = 'AIR / AIR'   | 'AIR' in modalidad: True
  pol.upper() = 'SANTIAGO'         | 'SANTIAGO' in pol: True
  peso = 50.0                 | peso < 3000: True
  bultos = 1.0             | bultos > 100: False


In [14]:
# CELDA 13 — Widget definitivo, autocontenido, con cluster heurístico garantizado
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widgets de entrada
w_id2            = widgets.Text(value='DEMO-001', description='ID Despacho:')
w_proveedor2     = widgets.Text(value='AIRSEALOG', description='Proveedor Servicio:')
w_principal2     = widgets.Text(value='PROJAR', description='Proveedor Principal:')
w_agencia2       = widgets.Text(value='AVM ADUANERA', description='Agencia Aduana:')
w_pol2           = widgets.Dropdown(
    options=['VALENCIA','SAN ANTONIO','SANTIAGO','BARCELONA','COLOMBO','MIAMI','BRUSELAS','LEIXOES'],
    value='SANTIAGO', description='POL (origen):')
w_pod2           = widgets.Dropdown(options=['CALLAO','LIMA'], value='CALLAO', description='POD (destino):')
w_modalidad2     = widgets.Dropdown(options=['SEA / FCL','SEA / LCL','AIR / AIR'], value='AIR / AIR', description='Modalidad:')
w_incoterm2      = widgets.Dropdown(options=['GRUPO_C','GRUPO_E','GRUPO_F','GRUPO_D'], value='GRUPO_E', description='INCOTERM:')
w_contenedores2  = widgets.IntSlider(value=0, min=0, max=10, description='Contenedores:')
w_bultos2        = widgets.IntSlider(value=1, min=0, max=2000, step=10, description='Bultos:')
w_peso2          = widgets.IntSlider(value=50, min=0, max=100000, step=500, description='Peso (kg):')
w_categoria2     = widgets.Dropdown(options=['SUSTRATOS','PLANTAS','AGROQUIMICOS','SPAREPARTS','MAQUINARIA','E&E','#N/D'],
                                    value='SPAREPARTS', description='Categoría:')
w_fecha2         = widgets.Text(value='2026-08-15', description='Fecha ETA:')

boton2 = widgets.Button(description='🔮 Predecir', button_style='primary', layout=widgets.Layout(width='200px'))
output2 = widgets.Output()

# Diccionario de nombres de cluster (autocontenido)
NOMBRES_CLUSTER_FINAL = {
    0: "Marítimos europeos pequeños con alta cantidad de bultos",
    1: "Aéreos express bajo costo (courier Santiago-Lima)",
    2: "Aéreos de alto valor declarado",
    3: "Marítimos europeos de gran volumen (sustratos)",
    4: "Marítimos asiáticos multi-contenedor (Colombo-Callao)",
    5: "Aéreos rutinarios Santiago-Callao (perfil mayoritario)",
}

def on_predict_final(b):
    with output2:
        clear_output()
        despacho = {
            'id_despacho':         w_id2.value,
            'proveedor_servicio':  w_proveedor2.value,
            'proveedor_principal': w_principal2.value,
            'agencia_aduana':      w_agencia2.value,
            'acreedor':            w_agencia2.value,
            'pol':                 w_pol2.value,
            'pod':                 w_pod2.value,
            'modalidad':           w_modalidad2.value,
            'incoterm_familia':    w_incoterm2.value,
            'contenedores':        w_contenedores2.value,
            'bultos':              w_bultos2.value,
            'peso_kg':             w_peso2.value,
            'categoria':           w_categoria2.value,
            'fecha_eta':           w_fecha2.value,
            'tipo_contenedor':     '40 HC',
            'pais_origen':         w_pol2.value,
        }

        # Llamar a predecir_despacho para obtener regresión + clasificación
        res = predecir_despacho(despacho)

        # SOBREESCRIBIR el cluster con la heurística (forzado, sin ambigüedad)
        cluster_heuristico = asignar_cluster_heuristico(despacho)
        res['cluster_id'] = cluster_heuristico
        res['cluster_nombre'] = NOMBRES_CLUSTER_FINAL.get(cluster_heuristico, 'Sin nombre')

        # Output
        print(f"💰 COSTO TOTAL ESTIMADO: S/ {res['costo_total_estimado']:,}")
        print(f"   Rango 80% confianza:    S/ {res['costo_minimo_estimado']:,} — S/ {res['costo_maximo_estimado']:,}")
        print(f"   Perfil operativo:       Cluster {res['cluster_id']} — {res['cluster_nombre']}")
        print(f"\n📊 Desglose por concepto:")
        print(res['predicciones_por_concepto'].sort_values('costo_predicho_pen', ascending=False).to_string(index=False))

boton2.on_click(on_predict_final)

col_izq2 = widgets.VBox([w_id2, w_proveedor2, w_principal2, w_agencia2, w_pol2, w_pod2])
col_der2 = widgets.VBox([w_modalidad2, w_incoterm2, w_contenedores2, w_bultos2, w_peso2, w_categoria2, w_fecha2])
formulario2 = widgets.HBox([col_izq2, col_der2])

display(widgets.VBox([
    widgets.HTML("<h3>🛠️ Simulador de Predicción (versión final)</h3>"),
    widgets.HTML("<p>Versión definitiva con cluster heurístico garantizado.</p>"),
    formulario2,
    boton2,
    output2
]))

💰 COSTO TOTAL ESTIMADO: S/ 27,613
   Rango 80% confianza:    S/ 4,181 — S/ 46,163
   Perfil operativo:       Cluster 1 — Aéreos express bajo costo (courier Santiago-Lima)

📊 Desglose por concepto:
                 concepto  costo_predicho_pen  intervalo_inferior  intervalo_superior riesgo_desvio
TRANSPORTE_T1_CALLAO_LIMA                3878                 302                3742         MEDIO
       DERECHOS_IMPUESTOS                3702                 641               17788          ALTO
                 DESCARGA                3597                 411                2631         MEDIO
     SERVICIOS_LOGISTICOS                3348                 351                3028         MEDIO
   SOBRESTADIA_ALMACENAJE                2838                 280                6137          ALTO
      FLETE_INTERNACIONAL                2798                 262                3162          ALTO
 TRANSPORTE_T2_LIMA_FUNDO                2153                 573                4290          ALTO
   